# NMS radius ablation: 7.5 um (D7 default) vs. 5.0 um, paired against `chromatin_od_ranker_variant.ipynb`

**What this notebook is.** `chromatin_od_ranker_variant.ipynb` (this same directory) measured the
cost and precision effect of re-ranking a `max_peaks=100`-capped, post-NMS candidate pool by
`chromatin_od` (od51) instead of `tm_score`, with NMS running at the repo default radius,
`ev.MIDOG_RADIUS_UM = 7.5` um (**D7**). This notebook asks a different question about the same
pipeline: what happens if the NMS suppression radius is shrunk to **5.0 um**, holding everything
else -- including the evaluation match radius, still 7.5 um -- fixed? Two things are reported: how
much timing overhead the smaller radius adds to the base pipeline (both the `tm_score` and
`chromatin_od` rankers), and whether it buys higher precision at the fixed budgets K=10/20/30.

**This is not a new experiment design -- D7 already ran it, and closed it.**
`f5_nms_radius_ablation.py` ablated exactly this radius on the unbounded (pre-`max_peaks`) pool and
found "no benefit on either ranking axis, at a cost of 23-37% more candidates" (`DECISIONS.md` D7).
Its Gate 4 additionally found the *head* of a score-ranked list nearly radius-invariant: at K=20
over these same 14 ROIs, 7.5 um and 5.0 um produce "the same 127 true positives with 279 of 280
coordinates shared". D7's own "what would change my mind" asks whether that head-of-list
invariance survives at seed indices F5 did not run, or a pipeline configuration it did not test --
this notebook is exactly that second case: `max_peaks=100` applied before NMS, then re-ranked by
`chromatin_od`, is a combination D7 never measured. This is a legitimate re-check at a new
operating point, not a proposal to reopen D7 -- see the closing readout for whether the result
actually moves.

**5.0 um is not a free substitution -- it conflicts with a runtime invariant.**
`invariants.check_nms_radius` (D7, invariant 4) raises unless the NMS radius exactly equals
`evaluate.radius_px(mpp)` at its default 7.5 um argument, precisely to catch the *accidental*
`NMS_RADIUS_UM=5.0` override this repo has propagated by copy before. Passing `nms_radius=5.0`
straight into `cp.Arm` here would trip that guard and halt the run. `f5_nms_radius_ablation.py`
established the correct way to run this ablation without disabling the check: only the arm that is
genuinely running at 7.5 um carries `nms_radius=nms_radius_75` (so `check_nms_radius` still runs on
it, as a *positive control* that the harness itself is intact); the 5.0 um arms carry
`nms_radius=None`, which skips a check whose premise (radius == match radius) those arms are
deliberately built to violate. This notebook follows that same precedent.

**Branch structure -- forked at stage 5, not run as two separate passes.** Stages 1-4 (seed
refinement through threshold/`max_peaks` peak extraction) do not depend on the NMS radius at all,
so they run once per ROI, exactly as in `chromatin_od_ranker_variant.ipynb`. Stage 5 (NMS +
self-hit suppression) then runs **twice** -- once at `nms_radius_75` (the control, reproducing that
notebook exactly), once at `nms_radius_50` (the ablation) -- producing two independent post-NMS
pools from the *same* pre-NMS candidate list. Stage 6 (rank by `tm_score` or by `chromatin_od`)
then runs on each pool, giving **four** arms per ROI: `{tm_score, chromatin_od} x {7.5 um, 5.0
um}`. Running both radii from one shared stages-1-4 pass, rather than as two separate notebook
runs, is what makes the timing and precision deltas paired within an ROI rather than subject to
cross-run noise -- the original notebook already flagged its own timing as a "single-run point
estimate" on ROIs whose wall-clock varies for unexplained reasons.

**The seed-annulus check does not hold at 5.0 um, and that is a likely mechanism, not a bug.**
`chromatin_od_ranker_variant.ipynb`'s `seed_annulus_empty` check passes at 7.5 um only because the
NMS radius and the match radius coincide there: NMS clears a disc of exactly `match_radius` around
the retained self-peak, and the `SELF_HIT_RADIUS=5.0` px filter then removes that self-peak,
leaving a hole of exactly `match_radius` around the seed's own template with no survivors inside
it. At `nms_radius_50` (~19.7-22.1 px, well under `match_radius`'s ~29.6-33.1 px), candidates in
that 20-33 px annulus around the seed's own nucleus can survive both NMS and the self-hit filter.
Because the seed annotation is excluded from `gt_eval`, any such survivor is a structural false
positive on the seed's own nucleus -- and it sits adjacent to the template's own peak, so it likely
carries a high `tm_score`, and since it is on the same (already-selected) nucleus, plausibly a high
`od51` too. This is measured directly below (not asserted away) as `n_near_seed_r50`, and is a
candidate explanation for whatever the precision numbers below show.

**Everything else is held fixed at production's accepted defaults** (`D8_TEMPLATE_ANCHOR.md`'s
current template anchor, D9's `max_peaks=100`), identical to `chromatin_od_ranker_variant.ipynb`.
`MATCH_RADIUS_UM` (the evaluation/scoring radius) is **not** touched -- the task asks about the NMS
radius specifically, and decoupling it from the match radius is exactly the ablation D7 already
priced.

In [1]:
import gc
import sys
import time

import cv2
import numpy as np
import pandas as pd

sys.path.insert(0, '..')
from midog_utils import channels as ch
from midog_utils import chromatin as cm
from midog_utils import dataset as ds
from midog_utils import evaluate as ev
from midog_utils import find_and_suppress as fs
from midog_utils import seed_selection as ss
from midog_utils import template_match as tm
from midog_utils import compare as cp
from midog_utils.nms import nms_by_distance

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 220)

NB_T0 = time.time()

# ---------------------------------------------------------------------------------------
# Config -- identical to chromatin_od_ranker_variant.ipynb's cell 1, except the NMS radius,
# which now forks into a control (nms_radius_75, D7's default) and an ablation (nms_radius_50).
# MATCH_RADIUS_UM is untouched -- this notebook ablates the suppression radius, not the
# evaluation radius.
# ---------------------------------------------------------------------------------------
IMAGES_DIR = '../images/extra_valid'
SEED_INDEX = 0

CHANNEL = 'hematoxylin_od'
METHOD = cv2.TM_CCOEFF
PEAK_MIN_DISTANCE = 7
SELF_HIT_RADIUS = 5.0
DEEP_FLOOR_Z = -1.5           # unchanged -- same permissive extraction floor as the template
MAX_PEAKS = 100                # unchanged -- fixed in every arm, not ablated here

NMS_RADIUS_UM_CONTROL = ev.MIDOG_RADIUS_UM   # 7.5 -- D7's default; reproduces
                                               # chromatin_od_ranker_variant.ipynb exactly
NMS_RADIUS_UM_ABLATION = 5.0                  # this notebook's one deliberate change
MATCH_RADIUS_UM = ev.MIDOG_RADIUS_UM          # unchanged -- the evaluation radius is not the
                                               # thing under test

CFG = fs.FSConfig(channel=CHANNEL, base_size=tm.BASE_SIZE, scales=(1.0,),
                  n_angles=1, flips=(False,), peak_min_distance=PEAK_MIN_DISTANCE,
                  self_hit_radius=SELF_HIT_RADIUS)
BORDER = CFG.patch_size // 2
OTSU_WINDOW = tm.BASE_SIZE

# The ranking ablation carries over unchanged from chromatin_od_ranker_variant.ipynb: two
# rankers, 'tm_score' (free) and 'chromatin_od' (od51, cm.chromatin_density, window=51,
# darkest-10%), each now applied to two pools (one per NMS radius) instead of one.
OD51_WINDOW = 51                # tm.BASE_SIZE -- matches the accepted chromatin sweep's od51
OD_PAD_51 = OD51_WINDOW // 2    # 25 -- exactly covers a 51px window
VARIANT_LABEL = 'chromatin_od (od51) ranking, max_peaks=100'

BUDGETS_FOR_EVAL = (10, 20, 30)   # same restricted scope as chromatin_od_ranker_variant.ipynb
BUDGET = max(BUDGETS_FOR_EVAL)    # 30 -- rank-and-truncate target for every arm

MAX_PEAKS_ORACLE_TIMING_CSV = 'max_peaks_100_timing.csv'
MAX_PEAKS_ORACLE_PRECISION_CSV = 'max_peaks_100_precision.csv'
CHROM_OD_ORACLE_TIMING_CSV = 'chromatin_od_ranker_timing.csv'
CHROM_OD_ORACLE_PRECISION_CSV = 'chromatin_od_ranker_precision.csv'

print(f'BUDGETS_FOR_EVAL={BUDGETS_FOR_EVAL}, match radius = {MATCH_RADIUS_UM} um (unchanged), '
      f'NMS radius: control={NMS_RADIUS_UM_CONTROL} um, ablation={NMS_RADIUS_UM_ABLATION} um, '
      f'channel={CHANNEL}, MAX_PEAKS={MAX_PEAKS} (fixed, every arm)')
print(f'chromatin_od (od51): window={OD51_WINDOW} frac={cm.DEFAULT_FRAC} OD_PAD_51={OD_PAD_51}')

BUDGETS_FOR_EVAL=(10, 20, 30), match radius = 7.5 um (unchanged), NMS radius: control=7.5 um, ablation=5.0 um, channel=hematoxylin_od, MAX_PEAKS=100 (fixed, every arm)
chromatin_od (od51): window=51 frac=0.1 OD_PAD_51=25


## Helpers

In [2]:
def draw_seed_with_retry(pool, rng, check_fn):
    """Draw a row via `rng.integers`; on failure drop it and redraw on the same stream."""
    working, retries = pool.copy(), 0
    while len(working) > 0:
        idx = int(rng.integers(len(working)))
        row = working.iloc[idx]
        result = check_fn(row)
        if result is not None:
            return row, result, retries
        working = working.drop(working.index[idx])
        retries += 1
    raise ValueError('seed pool exhausted -- no candidate passed check_fn')


def suppress(centers, scores, radius, ref_xy):
    """NMS at `radius`, then drop the template's own self-correlation."""
    keep = nms_by_distance(centers, scores, radius)
    c, s = centers[keep], scores[keep]
    if len(c):
        ok = np.hypot(c[:, 0] - ref_xy[0], c[:, 1] - ref_xy[1]) > SELF_HIT_RADIUS
        c, s = c[ok], s[ok]
    return c, s


def roi_files(images_dir=IMAGES_DIR):
    import os
    return sorted(f for f in os.listdir(images_dir) if f.endswith('.tiff'))


print(f'{len(roi_files())} ROIs on disk in {IMAGES_DIR}/')

14 ROIs on disk in ../images/extra_valid/


## Per-ROI worker

In [3]:
def run_roi(fn, image_id, domain, anns):
    gc.disable()

    # =================== SETUP (once per ROI; not part of click latency) ==============
    t0 = time.perf_counter()
    path = f'{IMAGES_DIR}/{fn}'
    rgb = ds.load_roi(path)
    mpp = ds.roi_mpp(path)
    roi_shape = rgb.shape
    match_radius = ev.radius_px(mpp, MATCH_RADIUS_UM)
    nms_radius_75 = ev.radius_px(mpp, NMS_RADIUS_UM_CONTROL)
    nms_radius_50 = ev.radius_px(mpp, NMS_RADIUS_UM_ABLATION)
    hem = ch.to_channel(rgb, CHANNEL)
    gray_inv = ch.to_gray_inverted(rgb)
    H, W = hem.shape[:2]
    del rgb

    gt = ds.image_annotations(anns, fn)
    seed_pool, flagged = ss.agreement_pool(gt[gt['category_id'] == ds.MITOTIC])
    seed_pool = ss.border_filter(seed_pool, BORDER, roi_shape)
    rng = np.random.default_rng([SEED_INDEX, image_id])

    def _check(row):
        r = ss.tightened_template_box(gray_inv, float(row['cx']), float(row['cy']),
                                      otsu_window=OTSU_WINDOW)
        if r is None:
            return None
        if tm.read_padded_patch(hem, r[1], r[2], CFG.patch_size) is None:
            return None
        return r

    seed, seed_tpl_probe, n_retries = draw_seed_with_retry(seed_pool, rng, _check)
    seed_ann_id = int(seed['ann_id'])
    click_cx, click_cy = float(seed['cx']), float(seed['cy'])
    gt_eval = gt[gt['ann_id'] != seed_ann_id].reset_index(drop=True)
    n_gt = int((gt_eval['category_id'] == ds.MITOTIC).sum())
    t_setup = time.perf_counter() - t0

    # ============ PIPELINE stages 1-4 (radius-independent; run once) ===================
    # extract_peaks (stage 4, MAX_PEAKS-capped) runs entirely before NMS, so the NMS radius
    # cannot change anything up to and including this point -- there is one pre-NMS
    # candidate list, fed into NMS twice below.
    stages = {}
    t_outer0 = time.perf_counter()

    t1 = time.perf_counter()
    r = ss.tightened_template_box(gray_inv, click_cx, click_cy, otsu_window=OTSU_WINDOW)
    assert r is not None, f'{fn}: seed was pre-validated in SETUP but refused here'
    border_probe = tm.read_padded_patch(hem, r[1], r[2], CFG.patch_size)
    assert border_probe is not None, f'{fn}: seed was pre-validated in SETUP but unreadable here'
    assert r == seed_tpl_probe, f'{fn}: single-shot refinement diverged from the SETUP search'
    base_size, tpl_cx, tpl_cy = r
    tpl_xy = (float(tpl_cx), float(tpl_cy))
    stages['t1_refine_seed_box_s'] = time.perf_counter() - t1

    t2 = time.perf_counter()
    patch = tm.read_padded_patch(hem, *tpl_xy, CFG.patch_size)
    templates, _ = tm.build_augmentations(patch, base_size, CFG.scales, CFG.n_angles, CFG.flips)
    stages['t2_patch_template_build_s'] = time.perf_counter() - t2

    t3 = time.perf_counter()
    PAD = max((t.shape[0] - 1) // 2 for t in templates)
    hem_p = cv2.copyMakeBorder(hem, PAD, PAD, PAD, PAD, borderType=cv2.BORDER_REPLICATE)
    fused_p, _, valid_p = tm.fused_response(hem_p, templates, CFG.scale_normalize, method=METHOD)
    stages['t3_template_matching_s'] = time.perf_counter() - t3

    fused = fused_p[PAD:PAD + H, PAD:PAD + W]
    valid = valid_p[PAD:PAD + H, PAD:PAD + W]
    assert bool(valid.all()), f'{fn}: padding left part of the ROI unreachable (PAD={PAD})'
    med, mad = tm.robust_stats(fused, valid)

    t4 = time.perf_counter()
    cut = med + DEEP_FLOOR_Z * mad
    centers, scores = tm.extract_peaks(fused, valid, PEAK_MIN_DISTANCE, cut, MAX_PEAKS)
    stages['t4_threshold_peak_extraction_s'] = time.perf_counter() - t4
    n_peaks = len(centers)

    # ================= STAGE 5, forked: NMS + self-hit at each radius ==================
    t5a = time.perf_counter()
    c75, s75 = suppress(centers, scores, nms_radius_75, tpl_xy)
    stages['t5_nms_selfhit_r75_s'] = time.perf_counter() - t5a
    assert bool(np.all(np.diff(s75) <= 0)), f'{fn}: r75 post-NMS pool is not score-descending'

    t5b = time.perf_counter()
    c50, s50 = suppress(centers, scores, nms_radius_50, tpl_xy)
    stages['t5_nms_selfhit_r50_s'] = time.perf_counter() - t5b
    assert bool(np.all(np.diff(s50) <= 0)), f'{fn}: r50 post-NMS pool is not score-descending'

    pool75 = pd.DataFrame({'cx': c75[:, 0], 'cy': c75[:, 1], 'score': s75})
    pool50 = pd.DataFrame({'cx': c50[:, 0], 'cy': c50[:, 1], 'score': s50})
    n_detections_75, n_detections_50 = len(pool75), len(pool50)

    def _n_near_seed(pool):
        if not len(pool):
            return 0
        d_seed = np.hypot(pool['cx'] - tpl_xy[0], pool['cy'] - tpl_xy[1])
        return int((d_seed <= match_radius).sum())

    n_near_seed_75 = _n_near_seed(pool75)
    n_near_seed_50 = _n_near_seed(pool50)

    # ===== STAGE 6, forked x2: rank each pool by tm_score (baseline) and od51 (variant) ====
    # hem_pad depends only on hem (the whole ROI), never on the pool or the radius, so it is
    # computed and timed ONCE here and reused by both forks. Measuring it inside each fork
    # separately (an earlier version of this notebook did) pays cv2.copyMakeBorder on the same
    # full-ROI array twice per ROI and lets independent cache/allocator noise on the *second*
    # call masquerade as a radius effect -- exactly the kind of artifact this notebook's own
    # "single-run point estimate" caveat elsewhere warns about, so it is designed out here
    # instead of merely disclaimed.
    t6a = time.perf_counter()
    hem_pad = cv2.copyMakeBorder(hem, OD_PAD_51, OD_PAD_51, OD_PAD_51, OD_PAD_51, cv2.BORDER_REPLICATE)
    stages['t6a_od_pad_s'] = time.perf_counter() - t6a

    def _stage6(pool, tag):
        s = {}
        t6 = time.perf_counter()
        pool.sort_values('score', ascending=False, na_position='last', kind='mergesort').head(BUDGET)
        s[f't6_baseline_{tag}_s'] = time.perf_counter() - t6

        px = pool['cx'].to_numpy() + OD_PAD_51
        py = pool['cy'].to_numpy() + OD_PAD_51
        t6b = time.perf_counter()
        pool['od51'] = [cm.chromatin_density(hem_pad, x, y, window=OD51_WINDOW) for x, y in zip(px, py)]
        s[f't6b_od51_loop_{tag}_s'] = time.perf_counter() - t6b

        t6c = time.perf_counter()
        pool.sort_values('od51', ascending=False, na_position='last', kind='mergesort').head(BUDGET)
        s[f't6c_variant_rank_{tag}_s'] = time.perf_counter() - t6c

        tie, nan_rate = cp._tie_and_nan(pool['od51'])
        return s, tie, nan_rate

    s6_75, tie_75, nan_75 = _stage6(pool75, 'r75')
    s6_50, tie_50, nan_50 = _stage6(pool50, 'r50')
    stages.update(s6_75)
    stages.update(s6_50)
    del hem_pad

    t_outer_total = time.perf_counter() - t_outer0
    gc.enable()

    assert nan_75 == 0.0, f'{fn}: r75 od51 NaN rate {nan_75} -- OD_PAD_51 too small'
    assert nan_50 == 0.0, f'{fn}: r50 od51 NaN rate {nan_50} -- OD_PAD_51 too small'

    # ============== Four arms, two pools, one gt_eval per ROI =============
    # Only the r75 arms carry nms_radius, so only they run invariants.check_nms_radius
    # (a positive control -- see the intro markdown). The r50 arms are built at a radius
    # that deliberately does not equal the match radius, which is exactly what that check
    # exists to catch, so they carry nms_radius=None and skip it (f5_nms_radius_ablation.py's
    # precedent).
    tm_arm_75 = cp.Arm('tm_score', (lambda d=pool75: d), rank_key='score', seeded=True,
                       z=DEEP_FLOOR_Z, z_dependent=True, nms_radius=nms_radius_75,
                       caps=(MAX_PEAKS,), coverage_key=f'{fn}_r75',
                       extra={'nms_radius_tag': 'r75', 'nms_radius_um': NMS_RADIUS_UM_CONTROL})
    chromatin_arm_75 = cp.Arm('chromatin_od', (lambda d=pool75: d), rank_key='od51', seeded=True,
                              z=DEEP_FLOOR_Z, z_dependent=True, nms_radius=nms_radius_75,
                              caps=(MAX_PEAKS,), coverage_key=f'{fn}_r75',
                              extra={'nms_radius_tag': 'r75', 'nms_radius_um': NMS_RADIUS_UM_CONTROL})
    tm_arm_50 = cp.Arm('tm_score', (lambda d=pool50: d), rank_key='score', seeded=True,
                       z=DEEP_FLOOR_Z, z_dependent=True, nms_radius=None,
                       caps=(MAX_PEAKS,), coverage_key=f'{fn}_r50',
                       extra={'nms_radius_tag': 'r50', 'nms_radius_um': NMS_RADIUS_UM_ABLATION})
    chromatin_arm_50 = cp.Arm('chromatin_od', (lambda d=pool50: d), rank_key='od51', seeded=True,
                              z=DEEP_FLOOR_Z, z_dependent=True, nms_radius=None,
                              caps=(MAX_PEAKS,), coverage_key=f'{fn}_r50',
                              extra={'nms_radius_tag': 'r50', 'nms_radius_um': NMS_RADIUS_UM_ABLATION})

    ctx = dict(file_name=fn, tumor_type=domain, image_id=image_id, seed_ann_id=seed_ann_id,
              base_size=base_size, n_retries=n_retries,
              map_median=round(float(med), 5), mad_scale=round(float(mad), 5))
    checks = []
    ev_out = cp.evaluate_arms([tm_arm_75, chromatin_arm_75, tm_arm_50, chromatin_arm_50],
                              gt_eval, match_radius, roi_shape=(H, W), mpp=mpp,
                              budgets=BUDGETS_FOR_EVAL, context=ctx, checks=checks)
    # Hard assert, r75 only -- reproduces chromatin_od_ranker_variant.ipynb's own check
    # exactly. The r50 annulus is reported below instead of asserted (see intro markdown):
    # a nonzero n_near_seed_r50 is the expected signature of this ablation, not a defect.
    checks.append(dict(check='seed_annulus_empty_r75', label=fn, n_near_seed=n_near_seed_75,
                       passed=bool(n_near_seed_75 == 0)))

    del hem, gray_inv, hem_p, fused_p, valid_p, fused, valid, templates, patch
    del centers, scores, c75, s75, c50, s50
    gc.collect()

    t_shared = sum(stages[k] for k in ['t1_refine_seed_box_s', 't2_patch_template_build_s',
                                       't3_template_matching_s', 't4_threshold_peak_extraction_s'])

    def _totals(tag):
        t5 = stages[f't5_nms_selfhit_{tag}_s']
        t6_base = stages[f't6_baseline_{tag}_s']
        # t6a_od_pad_s is shared (paid once above, identical for both forks) -- included in
        # both totals below since either fork, run standalone, would still pay it once.
        t6_overhead = (stages['t6a_od_pad_s'] + stages[f't6b_od51_loop_{tag}_s']
                      + stages[f't6c_variant_rank_{tag}_s'])
        return dict(**{f't_chromatin_od_overhead_{tag}_s': round(t6_overhead, 5),
                       f't_baseline_pipeline_{tag}_s': round(t_shared + t5 + t6_base, 5),
                       f't_variant_pipeline_{tag}_s': round(t_shared + t5 + t6_overhead, 5)})

    totals = {}
    totals.update(_totals('r75'))
    totals.update(_totals('r50'))

    timing_row = dict(
        file_name=fn, tumor_type=domain, base_size=base_size, n_retries=n_retries,
        seed_ann_id=seed_ann_id, map_median=round(float(med), 5), mad_scale=round(float(mad), 5),
        n_peaks=n_peaks, n_detections_r75=n_detections_75, n_detections_r50=n_detections_50,
        n_near_seed_r75=n_near_seed_75, n_near_seed_r50=n_near_seed_50,
        t_setup_s=round(t_setup, 5),
        **{k: round(v, 5) for k, v in stages.items()},
        t_shared_stages_s=round(t_shared, 5),
        **totals,
        t_outer_total_s=round(t_outer_total, 5),
        chromatin_od_nan_rate_r75=nan_75, chromatin_od_largest_tie_block_r75=tie_75,
        chromatin_od_nan_rate_r50=nan_50, chromatin_od_largest_tie_block_r50=tie_50,
    )

    print(f"[{fn}] {domain:32s} base={base_size:2d} "
          f"n_det: r75={n_detections_75:3d} r50={n_detections_50:3d} "
          f"n_near_seed_r50={n_near_seed_50} "
          f"baseline: {timing_row['t_baseline_pipeline_r75_s']*1000:6.2f}->"
          f"{timing_row['t_baseline_pipeline_r50_s']*1000:6.2f}ms "
          f"variant: {timing_row['t_variant_pipeline_r75_s']*1000:6.2f}->"
          f"{timing_row['t_variant_pipeline_r50_s']*1000:6.2f}ms", flush=True)

    return timing_row, ev_out, checks

## Run -- all 14 ROIs

In [4]:
images, annotations = ds.load_annotations('../databases/MIDOG++.json')
ds.check_invariants(annotations)
meta_ix = images.set_index('file_name')[['image_id', 'tumor_type']]

files = roi_files()
assert len(files) == 14, f'expected 14 ROIs in {IMAGES_DIR}/, found {len(files)}'

timing_rows, all_evs, all_checks = [], [], []
t_run = time.time()
for fn in files:
    image_id = int(meta_ix.loc[fn, 'image_id'])
    domain = meta_ix.loc[fn, 'tumor_type']
    trow, ev_out, checks = run_roi(fn, image_id, domain, annotations)
    timing_rows.append(trow)
    all_evs.append(ev_out)
    all_checks.extend(checks)
    gc.collect()

TIMING = pd.DataFrame(timing_rows).set_index('file_name')
ALL_RAW = pd.concat(all_evs, ignore_index=True)
CHECKS = pd.DataFrame(all_checks)

print(f'\n{len(files)} ROIs timed in {time.time() - t_run:.0f}s')
print(f'checks passed: {int(CHECKS["passed"].sum())}/{len(CHECKS)}')
assert CHECKS['passed'].all(), 'a pipeline-invariant check failed -- see CHECKS above'
CHECKS['check'].value_counts()

[013.tiff] human breast cancer              base=31 n_det: r75= 97 r50= 98 n_near_seed_r50=0 baseline: 2272.32->2271.27ms variant: 2415.31->2413.49ms


[094.tiff] human breast cancer              base=25 n_det: r75= 96 r50= 96 n_near_seed_r50=0 baseline: 1694.21->1693.85ms variant: 1734.87->1734.59ms


[201.tiff] canine lung cancer               base=51 n_det: r75= 98 r50= 98 n_near_seed_r50=0 baseline: 1812.16->1811.84ms variant: 1846.17->1846.10ms


[233.tiff] canine lung cancer               base=25 n_det: r75= 97 r50= 99 n_near_seed_r50=0 baseline: 1453.41->1453.03ms variant: 1491.63->1491.71ms


[245.tiff] canine lymphosarcoma             base=47 n_det: r75= 89 r50= 98 n_near_seed_r50=0 baseline: 1727.70->1727.34ms variant: 1763.19->1763.52ms


...

[460.tiff] canine soft tissue sarcoma       base=47 n_det: r75= 99 r50= 99 n_near_seed_r50=0 baseline: 1299.77->1299.43ms variant: 1329.66->1329.00ms


[529.tiff] human melanoma                   base=37 n_det: r75= 97 r50= 98 n_near_seed_r50=0 baseline: 1372.68->1372.35ms variant: 1405.28->1405.05ms


[548.tiff] human melanoma                   base=29 n_det: r75= 98 r50= 99 n_near_seed_r50=0 baseline: 1310.09->1309.95ms variant: 1346.68->1345.80ms



14 ROIs timed in 73s
checks passed: 98/98


check
no_cap                    56
nms_radius                28
seed_annulus_empty_r75    14
Name: count, dtype: int64

## Verification 1 -- tm_score branch @ 7.5 um reproduces `max_peaks_100_variant.ipynb`

`max_peaks_100_variant.ipynb`'s own `variant` branch (`max_peaks=100`, ranked by `tm_score`, NMS at
7.5 um) is exactly this notebook's `tm_score` arm at `nms_radius_tag='r75'` -- the same cap, the
same radius, the same ranker, the same 14 ROIs and seed. If this fails, nothing below can be
trusted.

In [5]:
oracle_timing = pd.read_csv(MAX_PEAKS_ORACLE_TIMING_CSV).set_index('file_name')
oracle_precision = pd.read_csv(MAX_PEAKS_ORACLE_PRECISION_CSV)
oracle_precision_variant = oracle_precision[oracle_precision['branch'] == 'variant'].copy()

oracle_roi = (oracle_timing[['seed_ann_id', 'base_size', 'n_detections_variant',
                             'map_median', 'mad_scale']]
             .rename(columns={'n_detections_variant': 'n_detections'}))
this_roi = TIMING[['seed_ann_id', 'base_size', 'map_median', 'mad_scale']].copy()
this_roi['n_detections'] = TIMING['n_detections_r75']

cmp = this_roi.join(oracle_roi, lsuffix='_this', rsuffix='_oracle')
mismatches = []
for col in ['seed_ann_id', 'base_size', 'n_detections']:
    bad = cmp[f'{col}_this'].astype(int) != cmp[f'{col}_oracle'].astype(int)
    if bad.any():
        mismatches.append((col, cmp.index[bad].tolist()))
for col in ['map_median', 'mad_scale']:
    bad = ~np.isclose(cmp[f'{col}_this'], cmp[f'{col}_oracle'], rtol=0, atol=1e-5)
    if bad.any():
        mismatches.append((col, cmp.index[bad].tolist()))

oracle_tp = oracle_precision_variant.pivot_table(index='file_name', values='tp_at_budget', columns='budget')
this_tm_tp = (ALL_RAW[(ALL_RAW['arm'] == 'tm_score') & (ALL_RAW['nms_radius_tag'] == 'r75')]
             .pivot_table(index='file_name', values='tp_at_budget', columns='budget'))
for k in BUDGETS_FOR_EVAL:
    this_k = this_tm_tp[k]
    oracle_k = oracle_tp.loc[this_k.index, k]
    bad = this_k.astype(int) != oracle_k.astype(int)
    if bad.any():
        mismatches.append((f'tp_at_{k}', this_k.index[bad].tolist()))

if mismatches:
    print('!! MISMATCH vs. max_peaks_100_variant.ipynb -- tm_score @ 7.5 um branch diverged:')
    for col, rois in mismatches:
        print(f'   {col}: {rois}')
    display(cmp)
    raise AssertionError("tm_score @ 7.5 um does not reproduce max_peaks_100_variant.ipynb's variant branch")

print(f"All {len(cmp)} ROIs' tm_score @ 7.5 um branch matches "
      f"{MAX_PEAKS_ORACLE_TIMING_CSV}/{MAX_PEAKS_ORACLE_PRECISION_CSV} exactly on "
      f"seed_ann_id, base_size, n_detections, map_median, mad_scale, and tp_at_budget "
      f"for K in {BUDGETS_FOR_EVAL}.")

All 14 ROIs' tm_score @ 7.5 um branch matches max_peaks_100_timing.csv/max_peaks_100_precision.csv exactly on seed_ann_id, base_size, n_detections, map_median, mad_scale, and tp_at_budget for K in (10, 20, 30).


## Verification 2 -- both arms @ 7.5 um reproduce `chromatin_od_ranker_variant.ipynb` exactly

A stronger, more direct check specific to this fork: since stages 1-4 and the r75 fork of stage 5
are architecturally identical to `chromatin_od_ranker_variant.ipynb`'s only pipeline, **both**
arms at `nms_radius_tag='r75'` -- `tm_score` (its `baseline`) and `chromatin_od` (its `variant`) --
should reproduce that notebook's already-committed `chromatin_od_ranker_timing.csv` /
`chromatin_od_ranker_precision.csv` bit-for-bit on every quantity NMS radius cannot touch. This
isolates the one thing that changed to exactly the r50 arms, not "some code got rewritten
somewhere in between".

In [6]:
chrom_oracle_timing = pd.read_csv(CHROM_OD_ORACLE_TIMING_CSV).set_index('file_name')
chrom_oracle_precision = pd.read_csv(CHROM_OD_ORACLE_PRECISION_CSV)

mismatches2 = []
for col, this_col in [('base_size', 'base_size'), ('seed_ann_id', 'seed_ann_id'),
                      ('n_detections', 'n_detections_r75')]:
    bad = chrom_oracle_timing[col].astype(int) != TIMING.loc[chrom_oracle_timing.index, this_col].astype(int)
    if bad.any():
        mismatches2.append((col, chrom_oracle_timing.index[bad].tolist()))
for col, this_col in [('map_median', 'map_median'), ('mad_scale', 'mad_scale')]:
    bad = ~np.isclose(chrom_oracle_timing[col],
                      TIMING.loc[chrom_oracle_timing.index, this_col], rtol=0, atol=1e-5)
    if bad.any():
        mismatches2.append((col, chrom_oracle_timing.index[bad].tolist()))

oracle_arm_map = {'baseline': 'tm_score', 'variant': 'chromatin_od'}
for oracle_branch, arm_name in oracle_arm_map.items():
    oracle_tp = (chrom_oracle_precision[chrom_oracle_precision['branch'] == oracle_branch]
                .pivot_table(index='file_name', values='tp_at_budget', columns='budget'))
    this_tp = (ALL_RAW[(ALL_RAW['arm'] == arm_name) & (ALL_RAW['nms_radius_tag'] == 'r75')]
              .pivot_table(index='file_name', values='tp_at_budget', columns='budget'))
    for k in BUDGETS_FOR_EVAL:
        bad = this_tp[k].astype(int) != oracle_tp.loc[this_tp.index, k].astype(int)
        if bad.any():
            mismatches2.append((f'{arm_name}_tp_at_{k}', this_tp.index[bad].tolist()))

if mismatches2:
    print('!! MISMATCH vs. chromatin_od_ranker_variant.ipynb -- r75 fork diverged:')
    for col, rois in mismatches2:
        print(f'   {col}: {rois}')
    raise AssertionError('the r75 fork does not reproduce chromatin_od_ranker_variant.ipynb exactly')

print(f"All {len(chrom_oracle_timing)} ROIs' r75 fork (both tm_score and chromatin_od) matches "
      f"{CHROM_OD_ORACLE_TIMING_CSV}/{CHROM_OD_ORACLE_PRECISION_CSV} exactly -- the only thing "
      f"this notebook changes is what happens at nms_radius_tag='r50'.")

All 14 ROIs' r75 fork (both tm_score and chromatin_od) matches chromatin_od_ranker_timing.csv/chromatin_od_ranker_precision.csv exactly -- the only thing this notebook changes is what happens at nms_radius_tag='r50'.


## Sanity checks -- both NMS radii are actually live, and od51 stays defined

`nms_radius_50` must sit meaningfully below `nms_radius_75` and above `PEAK_MIN_DISTANCE` (the
local-maxima spacing already enforced at extraction) -- otherwise the "5.0 um" knob would be a
no-op relative to what stage 4 already guarantees. `od51` must also remain finite for every
survivor on both pools; `OD_PAD_51=25` was validated against a 60px pad on the unbounded pool, and
Verification 2 already confirms it still holds at 7.5 um on this capped pool -- this cell confirms
it also holds at 5.0 um, where the pool can only be the same size or larger.

In [7]:
mpp_by_roi = TIMING.index.to_series().apply(lambda fn: ds.roi_mpp(f'{IMAGES_DIR}/{fn}'))
radii_check = pd.DataFrame({
    'mpp': mpp_by_roi,
    'nms_radius_75_px': mpp_by_roi.apply(lambda m: ev.radius_px(m, NMS_RADIUS_UM_CONTROL)),
    'nms_radius_50_px': mpp_by_roi.apply(lambda m: ev.radius_px(m, NMS_RADIUS_UM_ABLATION)),
})
print(radii_check.round(2))
print(f"\nnms_radius_50_px ranges {radii_check['nms_radius_50_px'].min():.1f}-"
     f"{radii_check['nms_radius_50_px'].max():.1f} px -- well above PEAK_MIN_DISTANCE="
     f"{PEAK_MIN_DISTANCE}px (the knob is live) and below nms_radius_75_px's "
     f"{radii_check['nms_radius_75_px'].min():.1f}-{radii_check['nms_radius_75_px'].max():.1f} px.")

print()
print(TIMING[['n_detections_r75', 'n_detections_r50',
             'chromatin_od_nan_rate_r75', 'chromatin_od_nan_rate_r50']])
assert (TIMING['chromatin_od_nan_rate_r75'] == 0.0).all(), 'od51 has NaN values on the r75 pool'
assert (TIMING['chromatin_od_nan_rate_r50'] == 0.0).all(), 'od51 has NaN values on the r50 pool'
print('\nod51 is defined for every candidate in every ROI at both radii (nan_rate=0.0 throughout).')

            mpp  nms_radius_75_px  nms_radius_50_px
file_name                                          
013.tiff   0.23             33.14             22.09
094.tiff   0.23             32.63             21.75
201.tiff   0.25             30.22             20.15
233.tiff   0.25             30.22             20.15
245.tiff   0.25             30.22             20.15
246.tiff   0.25             30.22             20.15
300.tiff   0.25             29.61             19.74
301.tiff   0.25             29.61             19.74
402.tiff   0.23             33.14             22.09
403.tiff   0.23             33.06             22.04
459.tiff   0.25             30.22             20.15
460.tiff   0.25             30.22             20.15
529.tiff   0.23             33.06             22.04
548.tiff   0.23             32.98             21.98

nms_radius_50_px ranges 19.7-22.1 px -- well above PEAK_MIN_DISTANCE=7px (the knob is live) and below nms_radius_75_px's 29.6-33.1 px.

           n_detections_r75  n_

## The seed-annulus effect at 5.0 um

`chromatin_od_ranker_variant.ipynb` asserted `n_near_seed == 0` as a hard invariant, true at 7.5 um
by construction (see intro markdown). At 5.0 um that hole can have survivors in it: candidates
20-33 px from the seed's own template centre, which is exactly the geometry F5 traced on 301.tiff
("off-centre peaks 23-29 px away on the same nucleus"). Since the seed annotation is excluded from
`gt_eval`, every one of these is a structural false positive with no possible true-positive
credit -- this cell measures how many ROIs are affected and whether any actually reach the K=30
budget on either ranker, which is the only way they could move the precision numbers below.

In [8]:
n_affected = int((TIMING['n_near_seed_r50'] > 0).sum())
print(f'n_near_seed_r50 > 0 on {n_affected}/{len(TIMING)} ROIs:')
print(TIMING.loc[TIMING['n_near_seed_r50'] > 0, ['n_near_seed_r50', 'n_detections_r50']])

r50_rows = ALL_RAW[ALL_RAW['nms_radius_tag'] == 'r50']
budget30 = r50_rows[r50_rows['budget'] == BUDGET]
print(f"\nAt K={BUDGET}, r50 arms budget_delivered == n_detections_r50 whenever the pool is "
     f"budget-starved, so an annulus survivor reaches the list only if it out-ranks enough of "
     f"the pool to land inside the top {BUDGET}. n_near_seed_r50 above already upper-bounds how "
     f"many candidates *could* contaminate a budget; it is a pool-membership count, not a "
     f"rank-within-top-{BUDGET} count -- treat the precision deltas in the Summary and closing "
     f"cells below as the direct measurement of whether they did.")

n_near_seed_r50 > 0 on 0/14 ROIs:
Empty DataFrame
Columns: [n_near_seed_r50, n_detections_r50]
Index: []

At K=30, r50 arms budget_delivered == n_detections_r50 whenever the pool is budget-starved, so an annulus survivor reaches the list only if it out-ranks enough of the pool to land inside the top 30. n_near_seed_r50 above already upper-bounds how many candidates *could* contaminate a budget; it is a pool-membership count, not a rank-within-top-30 count -- treat the precision deltas in the Summary and closing cells below as the direct measurement of whether they did.


## Table 1 -- per-ROI stage timing, 7.5 um vs. 5.0 um (ms)

In [9]:
STAGE_COLS_SHARED = ['t1_refine_seed_box_s', 't2_patch_template_build_s',
                    't3_template_matching_s', 't4_threshold_peak_extraction_s', 't6a_od_pad_s']
RADIUS_STAGE_COLS = {tag: [f't5_nms_selfhit_{tag}_s', f't6_baseline_{tag}_s',
                          f't6b_od51_loop_{tag}_s',
                          f't6c_variant_rank_{tag}_s'] for tag in ('r75', 'r50')}
TOTAL_COLS = {tag: [f't_chromatin_od_overhead_{tag}_s', f't_baseline_pipeline_{tag}_s',
                   f't_variant_pipeline_{tag}_s'] for tag in ('r75', 'r50')}
ALL_TIME_COLS = (['t_setup_s'] + STAGE_COLS_SHARED + ['t_shared_stages_s']
                 + RADIUS_STAGE_COLS['r75'] + TOTAL_COLS['r75']
                 + RADIUS_STAGE_COLS['r50'] + TOTAL_COLS['r50'] + ['t_outer_total_s'])

TIMING_MS = TIMING.copy()
for c in ALL_TIME_COLS:
    TIMING_MS[c[:-2] + '_ms'] = (TIMING_MS[c] * 1000).round(2)

display_cols = (['tumor_type', 'base_size', 'n_detections_r75', 'n_detections_r50',
                't_setup_ms'] + [c[:-2] + '_ms' for c in STAGE_COLS_SHARED]
               + [c[:-2] + '_ms' for c in RADIUS_STAGE_COLS['r75']]
               + [c[:-2] + '_ms' for c in RADIUS_STAGE_COLS['r50']]
               + [c[:-2] + '_ms' for c in TOTAL_COLS['r75']]
               + [c[:-2] + '_ms' for c in TOTAL_COLS['r50']])
TIMING_MS.to_csv('chromatin_od_ranker_nms5um_timing.csv')
print('-> chromatin_od_ranker_nms5um_timing.csv')
TIMING_MS[display_cols]

-> chromatin_od_ranker_nms5um_timing.csv


,tumor_type,base_size,n_detections_r75,n_detections_r50,t_setup_ms,t1_refine_seed_box_ms,t2_patch_template_build_ms,t3_template_matching_ms,t4_threshold_peak_extraction_ms,t6a_od_pad_ms,t5_nms_selfhit_r75_ms,t6_baseline_r75_ms,t6b_od51_loop_r75_ms,t6c_variant_rank_r75_ms,t5_nms_selfhit_r50_ms,t6_baseline_r50_ms,t6b_od51_loop_r50_ms,t6c_variant_rank_r50_ms,t_chromatin_od_overhead_r75_ms,t_baseline_pipeline_r75_ms,t_variant_pipeline_r75_ms,t_chromatin_od_overhead_r50_ms,t_baseline_pipeline_r50_ms,t_variant_pipeline_r50_ms
file_name,,,,,,,,,,,,,,,,,,,,,,,,
013.tiff,human breast cancer,31,97,98,5344.12,1.55,0.03,1897.72,370.89,136.62,1.52,0.61,6.45,0.53,0.81,0.27,5.48,0.38,143.59,2272.32,2415.31,142.49,2271.27,2413.49
094.tiff,human breast cancer,25,96,96,3737.45,1.10,0.03,1342.00,349.82,35.61,0.84,0.41,5.00,0.46,0.65,0.23,4.98,0.38,41.07,1694.21,1734.87,40.97,1693.85,1734.59
201.tiff,canine lung cancer,51,98,98,3161.62,1.55,0.03,1511.01,298.33,28.68,0.84,0.40,5.36,0.37,0.70,0.23,5.44,0.37,34.42,1812.16,1846.17,34.49,1811.84,1846.10
233.tiff,canine lung cancer,25,97,99,3187.38,1.73,0.03,1152.75,297.65,32.92,0.82,0.41,5.34,0.38,0.63,0.23,5.47,0.52,38.64,1453.41,1491.63,38.91,1453.03,1491.71
245.tiff,canine lymphosarcoma,47,89,98,3274.43,1.57,0.03,1421.21,303.65,29.74,0.83,0.41,5.75,0.41,0.64,0.23,6.28,0.40,35.89,1727.70,1763.19,36.41,1727.34,1763.52
246.tiff,canine lymphosarcoma,41,96,98,3111.23,1.69,0.03,1273.92,283.84,27.61,0.80,0.39,4.93,0.41,0.61,0.22,4.92,0.33,32.95,1560.66,1593.22,32.86,1560.29,1592.94
300.tiff,canine cutaneous mast cell tumor,45,98,98,2803.15,2.12,0.03,1200.45,247.51,25.97,0.74,0.36,5.10,0.36,0.56,0.20,4.84,0.34,31.43,1451.21,1482.28,31.15,1450.86,1481.82
301.tiff,canine cutaneous mast cell tumor,41,98,98,2640.67,1.28,0.03,1069.54,242.34,23.91,0.69,0.35,4.26,0.32,0.52,0.19,3.98,0.28,28.49,1314.24,1342.38,28.17,1313.89,1341.88
402.tiff,human neuroendocrine tumor,29,98,99,3051.28,0.91,0.02,1137.99,286.32,31.39,0.70,0.34,4.07,0.29,0.49,0.27,3.90,0.28,35.76,1426.27,1461.69,35.57,1425.99,1461.30


In [10]:
print('Shared stages (radius-independent by construction -- stages 1-4 plus t6a_od_pad, '
     'which pads the whole ROI once regardless of pool size or radius):')
for col in [c[:-2] + '_ms' for c in STAGE_COLS_SHARED]:
    print(f'  {col:34s} mean={TIMING_MS[col].mean():7.2f}ms')

print()
print('Stage 5 (NMS + self-hit), 7.5 um vs. 5.0 um:')
d5 = TIMING_MS['t5_nms_selfhit_r50_ms'].mean() - TIMING_MS['t5_nms_selfhit_r75_ms'].mean()
print(f"  r75: {TIMING_MS['t5_nms_selfhit_r75_ms'].mean():7.3f}ms mean   "
     f"r50: {TIMING_MS['t5_nms_selfhit_r50_ms'].mean():7.3f}ms mean   delta={d5:+.3f}ms")

print()
print('Stage 6b+6c only (od51 loop + resort -- the part that actually scales with pool size; '
     't6a is shared above and cancels out of this delta by construction):')
d6bc = ((TIMING_MS['t6b_od51_loop_r50_ms'] + TIMING_MS['t6c_variant_rank_r50_ms'])
       - (TIMING_MS['t6b_od51_loop_r75_ms'] + TIMING_MS['t6c_variant_rank_r75_ms']))
print(f'  r75: {(TIMING_MS["t6b_od51_loop_r75_ms"] + TIMING_MS["t6c_variant_rank_r75_ms"]).mean():7.3f}ms mean   '
     f'r50: {(TIMING_MS["t6b_od51_loop_r50_ms"] + TIMING_MS["t6c_variant_rank_r50_ms"]).mean():7.3f}ms mean   '
     f'delta={d6bc.mean():+.3f}ms')

print()
for col75, col50, label in [('t_baseline_pipeline_r75_ms', 't_baseline_pipeline_r50_ms', 'tm_score'),
                           ('t_chromatin_od_overhead_r75_ms', 't_chromatin_od_overhead_r50_ms',
                            'chromatin_od stage-6 (t6a+t6b+t6c)')]:
    m75, m50 = TIMING_MS[col75].mean(), TIMING_MS[col50].mean()
    print(f'{label:34s}: r75={m75:7.3f}ms  r50={m50:7.3f}ms  delta={m50 - m75:+.3f}ms '
         f'({(m50 - m75) / m75 * 100:+.1f}%)')

print()
for tag in ('r75', 'r50'):
    b = TIMING_MS[f't_baseline_pipeline_{tag}_ms'].mean()
    v = TIMING_MS[f't_variant_pipeline_{tag}_ms'].mean()
    print(f'Full pipeline @ {tag}: baseline={b:7.2f}ms  variant={v:7.2f}ms  '
         f'delta={v - b:+.2f}ms ({(v - b) / b * 100:+.1f}%)')

Shared stages (radius-independent by construction -- stages 1-4 plus t6a_od_pad, which pads the whole ROI once regardless of pool size or radius):
  t1_refine_seed_box_ms              mean=   1.36ms
  t2_patch_template_build_ms         mean=   0.03ms
  t3_template_matching_ms            mean=1264.50ms
  t4_threshold_peak_extraction_ms    mean= 289.15ms
  t6a_od_pad_ms                      mean=  42.41ms

Stage 5 (NMS + self-hit), 7.5 um vs. 5.0 um:
  r75:   0.793ms mean   r50:   0.586ms mean   delta=-0.207ms

Stage 6b+6c only (od51 loop + resort -- the part that actually scales with pool size; t6a is shared above and cancels out of this delta by construction):
  r75:   5.282ms mean   r50:   5.054ms mean   delta=-0.228ms

tm_score                          : r75=1556.213ms  r50=1555.830ms  delta=-0.383ms (-0.0%)
chromatin_od stage-6 (t6a+t6b+t6c): r75= 47.688ms  r50= 47.461ms  delta=-0.226ms (-0.5%)

Full pipeline @ r75: baseline=1556.21ms  variant=1603.52ms  delta=+47.30ms (+3.0%)
Full 

## Table 2 -- precision@{10,20,30}, branch x NMS radius

In [11]:
ALL_RAW['branch'] = ALL_RAW['arm'].map({'tm_score': 'baseline', 'chromatin_od': 'variant'})
ALL_RAW['precision_at_budget'] = (ALL_RAW['tp_at_budget']
                                  / ALL_RAW['budget_delivered'].replace(0, np.nan))

PRECISION_LONG = ALL_RAW[['file_name', 'tumor_type', 'branch', 'arm', 'nms_radius_tag', 'budget',
                          'n_detections', 'n_gt_mitotic', 'budget_delivered', 'tp_at_budget',
                          'precision_at_budget', 'recall_at_budget']].copy()
PRECISION_LONG = PRECISION_LONG.sort_values(
    ['file_name', 'budget', 'nms_radius_tag', 'branch']).reset_index(drop=True)
PRECISION_LONG.to_csv('chromatin_od_ranker_nms5um_precision.csv', index=False)
print(f'-> chromatin_od_ranker_nms5um_precision.csv  ({len(PRECISION_LONG)} rows = 14 ROIs x '
     f'2 branches x 2 radii x {len(BUDGETS_FOR_EVAL)} budgets)')

for tag in ('r75', 'r50'):
    for branch in ['baseline', 'variant']:
        sub = PRECISION_LONG[(PRECISION_LONG['nms_radius_tag'] == tag)
                             & (PRECISION_LONG['branch'] == branch)
                             & (PRECISION_LONG['budget'] == BUDGET)]
        n_starved = int((sub['budget_delivered'] < BUDGET).sum())
        print(f'{tag} {branch:10s}: delivers fewer than BUDGET={BUDGET} candidates on '
             f'{n_starved}/{len(sub)} ROIs')

PRECISION_LONG.round(4)

-> chromatin_od_ranker_nms5um_precision.csv  (168 rows = 14 ROIs x 2 branches x 2 radii x 3 budgets)
r75 baseline  : delivers fewer than BUDGET=30 candidates on 0/14 ROIs
r75 variant   : delivers fewer than BUDGET=30 candidates on 0/14 ROIs
r50 baseline  : delivers fewer than BUDGET=30 candidates on 0/14 ROIs
r50 variant   : delivers fewer than BUDGET=30 candidates on 0/14 ROIs


,file_name,tumor_type,branch,arm,nms_radius_tag,budget,n_detections,n_gt_mitotic,budget_delivered,tp_at_budget,precision_at_budget,recall_at_budget
0,013.tiff,human breast cancer,baseline,tm_score,r50,10,98,17,10,3,0.3000,0.1765
1,013.tiff,human breast cancer,variant,chromatin_od,r50,10,98,17,10,5,0.5000,0.2941
2,013.tiff,human breast cancer,baseline,tm_score,r75,10,97,17,10,3,0.3000,0.1765
3,013.tiff,human breast cancer,variant,chromatin_od,r75,10,97,17,10,6,0.6000,0.3529
4,013.tiff,human breast cancer,baseline,tm_score,r50,20,98,17,20,4,0.2000,0.2353
5,013.tiff,human breast cancer,variant,chromatin_od,r50,20,98,17,20,9,0.4500,0.5294
6,013.tiff,human breast cancer,baseline,tm_score,r75,20,97,17,20,4,0.2000,0.2353
7,013.tiff,human breast cancer,variant,chromatin_od,r75,20,97,17,20,9,0.4500,0.5294
8,013.tiff,human breast cancer,baseline,tm_score,r50,30,98,17,30,6,0.2000,0.3529
9,013.tiff,human breast cancer,variant,chromatin_od,r50,30,98,17,30,9,0.3000,0.5294


## Table 2b -- precision@K pivoted for readability

In [12]:
PRECISION_PIVOT = PRECISION_LONG.pivot_table(
    index=['tumor_type', 'file_name'], columns=['budget', 'nms_radius_tag', 'branch'],
    values='precision_at_budget'
)
PRECISION_PIVOT.round(4)

budget                                           10                                20                                30                         
nms_radius_tag                                  r50              r75              r50              r75              r50              r75        
branch                                     baseline variant baseline variant baseline variant baseline variant baseline variant baseline variant
tumor_type                       file_name                                                                                                      
canine cutaneous mast cell tumor 300.tiff       0.7     0.9      0.7     0.9     0.70    0.90     0.70    0.90   0.6667  0.8667   0.6667  0.8667
                                 301.tiff       0.9     0.8      0.9     0.8     0.90    0.90     0.90    0.90   0.8000  0.9000   0.8000  0.9000
canine lung cancer               201.tiff       0.3     0.4      0.3     0.4     0.25    0.40     0.25    0.40   0.2000  0.3000   0.2000  0.3000
                                 233.tiff       0.3     0.7      0.3     0.7     0.25    0.40     0.25    0.40   0.2667  0.3000   0.2667  0.3333
canine lymphosarcoma             245.tiff       0.1     0.2      0.1     0.2     0.10    0.15     0.10    0.10   0.1333  0.1000   0.1000  0.0667
                                 246.tiff       1.0     1.0      1.0     1.0     0.90    0.80     0.90    0.70   0.7667  0.6667   0.8000  0.7000
canine soft tissue sarcoma       459.tiff       0.6     0.9      0.6     0.9     0.55    0.85     0.55    0.85   0.6333  0.8333   0.6333  0.8333
                                 460.tiff       0.7     0.7      0.7     0.7     0.50    0.80     0.50    0.80   0.4000  0.5667   0.4000  0.5667
human breast cancer              013.tiff       0.3     0.5      0.3     0.6     0.20    0.45     0.20    0.45   0.2000  0.3000   0.2000  0.3000
                                 094.tiff       0.4     0.5      0.4     0.5     0.50    0.75     0.50    0.75   0.5333  0.7333   0.5333  0.7333
human melanoma                   529.tiff       0.3     0.6      0.3     0.6     0.20    0.35     0.20    0.35   0.1333  0.2333   0.1333  0.2333
                                 548.tiff       0.5     0.7      0.5     0.8     0.50    0.60     0.50    0.65   0.5333  0.6667   0.5333  0.7000
human neuroendocrine tumor       402.tiff       0.4     0.6      0.4     0.7     0.45    0.55     0.45    0.60   0.4667  0.5000   0.4667  0.5000
                                 403.tiff       0.5     0.5      0.5     0.5     0.35    0.50     0.35    0.50   0.2667  0.4667   0.2667  0.4667

## Summary -- pooled precision and win counts, by budget, branch and radius

In [13]:
print('Pooled precision (sum tp / sum delivered) across all 14 ROIs:')
for k in BUDGETS_FOR_EVAL:
    print(f'  K={k}:')
    for tag in ('r75', 'r50'):
        for branch in ['baseline', 'variant']:
            sub = PRECISION_LONG[(PRECISION_LONG['nms_radius_tag'] == tag)
                                 & (PRECISION_LONG['branch'] == branch)
                                 & (PRECISION_LONG['budget'] == k)]
            pooled = sub['tp_at_budget'].sum() / sub['budget_delivered'].sum()
            print(f'    {tag} {branch:10s}: {pooled:.4f}  ({int(sub["tp_at_budget"].sum())} tp / '
                 f'{int(sub["budget_delivered"].sum())} delivered)')

print()
print('Win counts, r50 vs. r75, same branch and budget (per-ROI precision delta):')
for k in BUDGETS_FOR_EVAL:
    for branch in ['baseline', 'variant']:
        r75 = PRECISION_LONG[(PRECISION_LONG['nms_radius_tag'] == 'r75')
                             & (PRECISION_LONG['branch'] == branch)
                             & (PRECISION_LONG['budget'] == k)].set_index('file_name')
        r50 = PRECISION_LONG[(PRECISION_LONG['nms_radius_tag'] == 'r50')
                             & (PRECISION_LONG['branch'] == branch)
                             & (PRECISION_LONG['budget'] == k)].set_index('file_name')
        d = r50['precision_at_budget'] - r75['precision_at_budget']
        r_lower = (r50['recall_at_budget'] < r75['recall_at_budget']).sum()
        print(f'K={k:2d} {branch:8s}: r50 precision delta: {int((d > 0).sum())} up / '
             f'{int((d < 0).sum())} down / {int((d == 0).sum())} unchanged of 14; '
             f'recall lower on {int(r_lower)}/14')

Pooled precision (sum tp / sum delivered) across all 14 ROIs:
  K=10:
    r75 baseline  : 0.5000  (70 tp / 140 delivered)
    r75 variant   : 0.6643  (93 tp / 140 delivered)
    r50 baseline  : 0.5000  (70 tp / 140 delivered)
    r50 variant   : 0.6429  (90 tp / 140 delivered)
  K=20:
    r75 baseline  : 0.4536  (127 tp / 280 delivered)
    r75 variant   : 0.5964  (167 tp / 280 delivered)
    r50 baseline  : 0.4536  (127 tp / 280 delivered)
    r50 variant   : 0.6000  (168 tp / 280 delivered)
  K=30:
    r75 baseline  : 0.4286  (180 tp / 420 delivered)
    r75 variant   : 0.5357  (225 tp / 420 delivered)
    r50 baseline  : 0.4286  (180 tp / 420 delivered)
    r50 variant   : 0.5310  (223 tp / 420 delivered)

Win counts, r50 vs. r75, same branch and budget (per-ROI precision delta):
K=10 baseline: r50 precision delta: 0 up / 0 down / 14 unchanged of 14; recall lower on 0/14
K=10 variant : r50 precision delta: 0 up / 3 down / 11 unchanged of 14; recall lower on 3/14
K=20 baseline: r50 p

### 245.tiff -- the one ROI where the mechanism actually reaches the top of the list

`245.tiff` is not a random pick: per this project's own memory, it is the one ROI in these 14
where a mitotic pair sits 26.57 px apart -- close enough for the 7.5 um radius to merge the pair
into one surviving peak, but far enough apart that the 5.0 um radius keeps both. D7's own text
names 245.tiff (seed 2, a different seed than this notebook's `SEED_INDEX=0`) as one of the four
cells that loses a mitosis to the 7.5 um radius. `n_detections_r75=89` vs. `n_detections_r50=98`
here is the largest gap of any ROI in this run (9 candidates, vs. 0-2 everywhere else) --
consistent with recovering a previously-merged detection, not routine NMS jitter. The win-count
table above already shows this ROI is *not* silent: it is one of the "1 up" cells at K=30 for
`tm_score` and one of the "up" cells at K=20/K=30 for `chromatin_od`. The cell below checks
directly whether one of those 9 extra candidates is a genuine true positive that reaches the
fixed K=30 budget, on both rankers -- if so, this is the sharpest single-ROI evidence in this
notebook that D7's "head of the list barely notices" does not hold universally, even though it
holds in the pooled aggregate (the gain here is exactly cancelled by losses elsewhere, which is
why the pooled win-count table alone would hide it).

In [14]:
roi_245 = PRECISION_LONG[PRECISION_LONG['file_name'] == '245.tiff'].sort_values(
    ['nms_radius_tag', 'branch', 'budget'])
print(roi_245[['nms_radius_tag', 'branch', 'budget', 'n_detections', 'budget_delivered',
              'tp_at_budget', 'precision_at_budget', 'recall_at_budget']].to_string(index=False))

print()
for branch in ['baseline', 'variant']:
    r75_30 = roi_245[(roi_245['nms_radius_tag'] == 'r75') & (roi_245['branch'] == branch)
                     & (roi_245['budget'] == 30)]['tp_at_budget'].iloc[0]
    r50_30 = roi_245[(roi_245['nms_radius_tag'] == 'r50') & (roi_245['branch'] == branch)
                     & (roi_245['budget'] == 30)]['tp_at_budget'].iloc[0]
    verdict = 'gains a true positive at K=30' if r50_30 > r75_30 else (
        'loses a true positive at K=30' if r50_30 < r75_30 else 'unchanged at K=30')
    print(f'{branch:10s}: tp_at_30 {int(r75_30)} -> {int(r50_30)}  ({verdict})')

print()
print("245.tiff's n_detections_r75=89 is also the smallest post-NMS pool of any ROI in this run "
     "(next smallest is 96) -- the 7.5 um radius suppresses unusually hard here, exactly where "
     "D7's own accounting says it should. This is n=1: one ROI, one seed, and the pooled tables "
     "above show it does not generalise to the other 13 -- but it is a real, checkable instance "
     "of the recall-at-the-cap-radius cost reaching a fixed-budget precision number, which the "
     "pooled aggregate alone cannot show.")

nms_radius_tag   branch  budget  n_detections  budget_delivered  tp_at_budget  precision_at_budget  recall_at_budget
           r50 baseline      10            98                10             1             0.100000          0.011236
           r50 baseline      20            98                20             2             0.100000          0.022472
           r50 baseline      30            98                30             4             0.133333          0.044944
           r50  variant      10            98                10             2             0.200000          0.022472
           r50  variant      20            98                20             3             0.150000          0.033708
           r50  variant      30            98                30             3             0.100000          0.033708
           r75 baseline      10            89                10             1             0.100000          0.011236
           r75 baseline      20            89                20 

## Closing readout

In [15]:
print(f'NMS radius {NMS_RADIUS_UM_CONTROL} um -> {NMS_RADIUS_UM_ABLATION} um, mean over {len(TIMING)} ROIs:')
for branch, base_col, var_col in [('baseline (tm_score)', 't_baseline_pipeline', None),
                                  ('variant (chromatin_od)', 't_variant_pipeline', None)]:
    b75 = TIMING_MS[f'{base_col}_r75_ms'].mean()
    b50 = TIMING_MS[f'{base_col}_r50_ms'].mean()
    print(f'  {branch:24s} full pipeline: {b75:.2f}ms -> {b50:.2f}ms '
         f'({b50 - b75:+.2f}ms, {(b50 - b75) / b75 * 100:+.1f}%)')

print()
print('Precision, pooled across 14 ROIs (7.5 um -> 5.0 um):')
for k in BUDGETS_FOR_EVAL:
    for branch in ['baseline', 'variant']:
        r75 = PRECISION_LONG[(PRECISION_LONG['nms_radius_tag'] == 'r75')
                             & (PRECISION_LONG['branch'] == branch) & (PRECISION_LONG['budget'] == k)]
        r50 = PRECISION_LONG[(PRECISION_LONG['nms_radius_tag'] == 'r50')
                             & (PRECISION_LONG['branch'] == branch) & (PRECISION_LONG['budget'] == k)]
        p75 = r75['tp_at_budget'].sum() / r75['budget_delivered'].sum()
        p50 = r50['tp_at_budget'].sum() / r50['budget_delivered'].sum()
        print(f'  K={k:2d} {branch:8s}: {p75:.4f} -> {p50:.4f}  (delta {p50 - p75:+.4f})')

print()
print(f"Caveat (D5, restated): this is single-seed, n=14 -- one click per ROI, not the paired, "
     f"multi-seed sweep D5 itself sets as the bar for changing a production default. The 7.5 um "
     f"arms reproduce chromatin_od_ranker_variant.ipynb's already-committed numbers exactly "
     f"(Verifications 1-2); the 5.0 um numbers are new -- no notebook in this repo has measured "
     f"this exact pipeline (max_peaks=100, then chromatin_od re-rank) at a 5.0 um NMS radius "
     f"before.")
print(f"D7's prior: at K=20 on the unbounded pool ranked by tm_score, 7.5 um and 5.0 um produced "
     f"the same 127 true positives with 279/280 coordinates shared -- i.e. near radius-invariance "
     f"at the head of the list. Whether that holds here, on a max_peaks=100-capped pool re-ranked "
     f"by chromatin_od, is exactly what the win-count table above answers -- and per the seed-"
     f"annulus section, any divergence has a specific, checkable candidate mechanism rather than "
     f"being an unexplained shift.")
print(f'\nnotebook ran in {time.time() - NB_T0:.0f}s')

NMS radius 7.5 um -> 5.0 um, mean over 14 ROIs:
  baseline (tm_score)      full pipeline: 1556.21ms -> 1555.83ms (-0.38ms, -0.0%)
  variant (chromatin_od)   full pipeline: 1603.52ms -> 1603.08ms (-0.43ms, -0.0%)

Precision, pooled across 14 ROIs (7.5 um -> 5.0 um):
  K=10 baseline: 0.5000 -> 0.5000  (delta +0.0000)
  K=10 variant : 0.6643 -> 0.6429  (delta -0.0214)
  K=20 baseline: 0.4536 -> 0.4536  (delta +0.0000)
  K=20 variant : 0.5964 -> 0.6000  (delta +0.0036)
  K=30 baseline: 0.4286 -> 0.4286  (delta +0.0000)
  K=30 variant : 0.5357 -> 0.5310  (delta -0.0048)

Caveat (D5, restated): this is single-seed, n=14 -- one click per ROI, not the paired, multi-seed sweep D5 itself sets as the bar for changing a production default. The 7.5 um arms reproduce chromatin_od_ranker_variant.ipynb's already-committed numbers exactly (Verifications 1-2); the 5.0 um numbers are new -- no notebook in this repo has measured this exact pipeline (max_peaks=100, then chromatin_od re-rank) at a 5.0 um NM

## Added-step accounting -- overhead added by shrinking the NMS radius to 5.0 um

In [16]:
print('=== Time added by shrinking NMS radius from 7.5 um to 5.0 um ===')
for branch, col in [('baseline (tm_score)', 't_baseline_pipeline'),
                    ('variant (chromatin_od)', 't_variant_pipeline')]:
    d = TIMING[f'{col}_r50_s'] - TIMING[f'{col}_r75_s']
    print(f'{branch}:')
    print(f'  mean    = {d.mean()*1000:7.3f}ms')
    print(f'  median  = {d.median()*1000:7.3f}ms')
    print(f'  std     = {d.std()*1000:7.3f}ms')
    print(f'  min/max = {d.min()*1000:7.3f}ms / {d.max()*1000:7.3f}ms')
    print(f'  total over 14 ROIs = {d.sum()*1000:7.2f}ms')
    print()

print('Decomposed -- where the added time actually comes from. t6a_od_pad_s is shared between '
     'both radii (paid once per ROI -- see the run_roi cell), so it drops out of this delta by '
     'construction rather than needing to be disclaimed away:')
d_t5 = TIMING['t5_nms_selfhit_r50_s'] - TIMING['t5_nms_selfhit_r75_s']
d_pool = TIMING['n_detections_r50'] - TIMING['n_detections_r75']
d_t6b = TIMING['t6b_od51_loop_r50_s'] - TIMING['t6b_od51_loop_r75_s']
d_t6c = TIMING['t6c_variant_rank_r50_s'] - TIMING['t6c_variant_rank_r75_s']
print(f'  stage 5 (NMS+self-hit) itself:                mean={d_t5.mean()*1000:7.3f}ms  '
     f'(NMS runs on the same <=100-point pre-NMS list either way -- a smaller radius changes '
     f'which points survive, not how many comparisons the algorithm makes, at this list size)')
print(f'  pool size after NMS (n_detections_r50 - r75): mean={d_pool.mean():6.2f} candidates '
     f'(a smaller radius suppresses fewer neighbours, so more of the max_peaks=100 pre-NMS list '
     f'survives)')
print(f'  stage 6b (od51 loop, scales with pool size):  mean={d_t6b.mean()*1000:7.3f}ms  '
     f'-- the mechanical consequence of the larger pool above, since chromatin_density is called '
     f'once per surviving candidate')
print(f'  stage 6c (re-sort by od51, scales with pool size): mean={d_t6c.mean()*1000:7.3f}ms')
print(f'  sum of the three above vs. the measured variant-pipeline delta: '
     f'{(d_t5 + d_t6b + d_t6c).mean()*1000:7.3f}ms vs. '
     f'{(TIMING["t_variant_pipeline_r50_s"] - TIMING["t_variant_pipeline_r75_s"]).mean()*1000:7.3f}ms '
     f'(match confirms t_shared and t6a_od_pad_s -- identical in both totals by construction -- '
     f'contribute nothing to the delta)')

=== Time added by shrinking NMS radius from 7.5 um to 5.0 um ===
baseline (tm_score):
  mean    =  -0.383ms
  median  =  -0.350ms
  std     =   0.202ms
  min/max =  -1.050ms /  -0.140ms
  total over 14 ROIs =   -5.36ms

variant (chromatin_od):
  mean    =  -0.434ms
  median  =  -0.405ms
  std     =   0.500ms
  min/max =  -1.820ms /   0.330ms
  total over 14 ROIs =   -6.08ms

Decomposed -- where the added time actually comes from. t6a_od_pad_s is shared between both radii (paid once per ROI -- see the run_roi cell), so it drops out of this delta by construction rather than needing to be disclaimed away:
  stage 5 (NMS+self-hit) itself:                mean= -0.207ms  (NMS runs on the same <=100-point pre-NMS list either way -- a smaller radius changes which points survive, not how many comparisons the algorithm makes, at this list size)
  pool size after NMS (n_detections_r50 - r75): mean=  1.21 candidates (a smaller radius suppresses fewer neighbours, so more of the max_peaks=100 pre-NM

## Precision effect of shrinking the NMS radius to 5.0 um

In [17]:
print('=== Precision effect of shrinking NMS radius from 7.5 um to 5.0 um ===')
pooled = {}
for k in BUDGETS_FOR_EVAL:
    pooled[k] = {}
    for branch in ['baseline', 'variant']:
        r75 = PRECISION_LONG[(PRECISION_LONG['nms_radius_tag'] == 'r75')
                             & (PRECISION_LONG['branch'] == branch) & (PRECISION_LONG['budget'] == k)]
        r50 = PRECISION_LONG[(PRECISION_LONG['nms_radius_tag'] == 'r50')
                             & (PRECISION_LONG['branch'] == branch) & (PRECISION_LONG['budget'] == k)]
        p75 = r75['tp_at_budget'].sum() / r75['budget_delivered'].sum()
        p50 = r50['tp_at_budget'].sum() / r50['budget_delivered'].sum()
        d = (r50.set_index('file_name')['precision_at_budget']
            - r75.set_index('file_name')['precision_at_budget'])
        pooled[k][branch] = dict(p75=p75, p50=p50, delta=p50 - p75,
                                 up=int((d > 0).sum()), down=int((d < 0).sum()),
                                 unchanged=int((d == 0).sum()))
        pct = (p50 - p75) / p75 * 100 if p75 else float('nan')
        print(f'  K={k:2d} {branch:8s}: pooled precision {p75:.4f} -> {p50:.4f}  '
             f'(delta {p50 - p75:+.4f}, {pct:+.1f}%)  |  per-ROI: '
             f"{pooled[k][branch]['up']} up / {pooled[k][branch]['down']} down / "
             f"{pooled[k][branch]['unchanged']} unchanged of 14")

print()
n_affected = int((TIMING['n_near_seed_r50'] > 0).sum())
if n_affected > 0:
    print(f'Bottom line: shrinking the NMS radius to 5.0 um leaves the seed-annulus check '
         f'violated on {n_affected}/14 ROIs (see "The seed-annulus effect at 5.0 um" above) -- '
         f"structural false positives on the seed's own nucleus that cannot occur at the 7.5 um "
         f'default. Any precision gain at 5.0 um has to survive this contamination risk to be a '
         f'real finding, and any precision loss is consistent with it.')
else:
    print(f'Bottom line: at this seed index (SEED_INDEX={SEED_INDEX}), the seed-annulus '
         f'mechanism the intro flagged as a candidate explanation did not fire on any of the 14 '
         f'ROIs (n_near_seed_r50=0 throughout) -- so it explains none of the deltas above. The '
         f'tm_score arm is exactly unchanged at every budget *pooled* (delta +0.0000 throughout) '
         f"-- but that pools over real, cancelling per-ROI churn (the win-count table shows 1 up "
         f"/ 1 down at K=30), not 14 silent ROIs: 245.tiff -- this project's own documented "
         f"merged-mitotic-pair case -- gains a true positive at K=30 on *both* rankers when the "
         f"radius shrinks (see the dedicated cell above), and is exactly cancelled by a loss "
         f"elsewhere. D7 Gate 4's head-of-list radius-invariance finding holds in the pooled "
         f'number at this operating point too, but not because nothing moves at the ROI level -- '
         f'because what moves happens to net to zero over these 14 ROIs. The chromatin_od arm '
         f'moves by at most 3.2% pooled (down at K=10 and K=30, up at K=20) -- small, not '
         f'one-directional, and largely the same ordinary reordering from the ~1.2 extra '
         f'candidates/ROI the smaller radius admits into an already max_peaks=100-capped pool, '
         f'plus the same 245.tiff recovery visible in the baseline arm.')
print(f'Either way: no budget here shows a precision improvement large or consistent enough to '
     f'read as a real gain from shrinking the NMS radius, and the small variant-arm changes read '
     f"as a plausible pool-composition effect rather than a mechanism decoupled from D7's own "
     f'finding. Both directions are single-run, single-seed, n=14 point estimates -- the same '
     f'caveat the source notebook and D5 itself state -- and none of this reopens D7: doing that '
     f'needs the annulus/proposal-stage effect shown at the seed indices F5 itself did not run, '
     f'which this notebook -- one seed per ROI -- also does not provide.')

=== Precision effect of shrinking NMS radius from 7.5 um to 5.0 um ===
  K=10 baseline: pooled precision 0.5000 -> 0.5000  (delta +0.0000, +0.0%)  |  per-ROI: 0 up / 0 down / 14 unchanged of 14
  K=10 variant : pooled precision 0.6643 -> 0.6429  (delta -0.0214, -3.2%)  |  per-ROI: 0 up / 3 down / 11 unchanged of 14
  K=20 baseline: pooled precision 0.4536 -> 0.4536  (delta +0.0000, +0.0%)  |  per-ROI: 0 up / 0 down / 14 unchanged of 14
  K=20 variant : pooled precision 0.5964 -> 0.6000  (delta +0.0036, +0.6%)  |  per-ROI: 2 up / 2 down / 10 unchanged of 14
  K=30 baseline: pooled precision 0.4286 -> 0.4286  (delta +0.0000, +0.0%)  |  per-ROI: 1 up / 1 down / 12 unchanged of 14
  K=30 variant : pooled precision 0.5357 -> 0.5310  (delta -0.0048, -0.9%)  |  per-ROI: 1 up / 3 down / 10 unchanged of 14

Bottom line: at this seed index (SEED_INDEX=0), the seed-annulus mechanism the intro flagged as a candidate explanation did not fire on any of the 14 ROIs (n_near_seed_r50=0 throughout) -- s